In [1]:
import sys
sys.path.append('../../Simulate/')

from UtilityFunctions import retrieve_iupac

In [2]:
import subprocess
import numpy as np

from typing import Dict

In [3]:
class StreamWGSIM:
    '''
    stream WGSIM output for bisulfite reads generation
    :param str sim_cmd: WGSIM commands for simulation
    :param bool pair_end: pair_end or not
    :rtype None
    '''
    def __init__(self, sim_cmd: list = None, pair_end: bool = True):
        self.sim_cmd  = sim_cmd
        self.pair_end = pair_end


    def __iter__(self):
        wgsim = subprocess.Popen(self.sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
        sim_iter = iter(wgsim.stdout.readline, b'')

        line  = self.get_line(sim_iter) # line is None when EOF
        while line:
            # collect all variant lines on the contig, after that sim_iter points to read lines
            if line == "Contig Variant Start":
                variant_contig, variant_dict = self.collect_variants(sim_iter)
                yield variant_contig, variant_dict

            # collect read pairs
            for collect_flag, read_pair in self.collect_reads(sim_iter):
                if collect_flag: # {1: collect_reads, 0: swith to collect_vars or EOF}
                    yield False, read_pair
                else:
                    line = "Contig Variant Start" if isinstance(read_pair, list) else None
                    break


    def collect_variants(self, sim_iter):
        '''collect variant lines from stdout'''
        variant_dict = {}
        variant_info = {}

        while True:
            line = self.get_line(sim_iter)
            if line == 'Contig Variant End':
                return variant_info['chrom'], variant_dict

            variant_info = self.process_variant_line(line)
            if variant_info['pos']:
                assert variant_info['pos'] not in variant_dict
                variant_dict[variant_info['pos']] = variant_info


    def collect_reads(self, sim_iter):
        '''collect read lines from stdout'''
        skip_flag = not self.pair_end

        while True:
            line  = self.get_line(sim_iter)
            if not line: # EOF
                yield 0, None
            elif line == "Contig Variant Start": # switch to collect variants
                yield 0, []
            else:
                read1 = self.process_read_lines(sim_iter, line = line)
                read2 = self.process_read_lines(sim_iter, skip = skip_flag)
                yield 1, [read1, read2]


    @staticmethod
    def get_line(sim_iter):
        '''receive lines from console'''
        try:
            line = next(sim_iter).strip()
        except StopIteration:
            print("End of output\n")
            return None
        else:
            return line

    @staticmethod
    def process_variant_line(line: str) -> Dict:
        '''parse variant lines'''
        line_split = line.split('\t')

        try:
            chrom, pos, ref, alt, heter_flag = line_split
        except ValueError:
            return dict(chrom=line_split[0], pos = None)
        else:
            heter = heter_flag == '+'
            indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
            offset= indel * max(len(ref), len(alt))
            if indel:
                iupac  = None
            else:
                iupac  = retrieve_iupac(alt)
                alt    = list(set(iupac) - set(ref))[0]
            return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                        offset=offset, heter=heter, indel=indel, iupac=iupac)

    @staticmethod
    def process_read_lines(sim_iter, line = None, skip = False):
        '''parse read lines'''
        if skip:
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            next(sim_iter)
            return None

        if not line:
            line = next(sim_iter).strip()
        # header, seq, comment process
        read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line.split(' ')
        cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
        seq = np.frombuffer(next(sim_iter).strip().encode(), dtype=np.int8)
        _, start, end, cover_pos, n_sub, n_indel, insert_size, inner_dist, ofs= next(sim_iter).strip().split(':')
        ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
        ctx = np.frombuffer(next(sim_iter).strip().encode(), np.int8)
        return dict(read_id=read_id, pair=int(pair), qual = int(qual),
                    flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                    start=int(start), end=int(end), cover_pos=int(cover_pos),
                    n_sub=int(n_sub), n_indel=int(n_indel),
                    insert_size=int(insert_size), inner_dist=int(inner_dist),
                    cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

# Test for different situation

In [4]:
sim_command = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','2022',
               '-A','0.05','-h','0', '-m', '1','/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

sim_command_empty_fasta = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/empty_fasta']

sim_command_nonexist_fasta = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSBolt/bsbolt/External/WGSIM/empty_fasta']


sim_command_no_snp = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.00', '-N','1000',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

sim_command_small_N = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
               '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
               '-r','0.001', '-N','10',
               '-R','0.15','-X','0.15',
               '-S','-1',
               '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

In [5]:
i = 0
for variant_contig, sim_data in StreamWGSIM(sim_command_no_snp):
    if variant_contig:
        print(sim_data)
    
    if variant_contig:
        print(variant_contig)
    if not variant_contig:
        [sim_data[0]['read_id'], sim_data[0]['pair']]
        ++i

/home/wbguo/iproject/BSReadSim/WGSIM/wgsim: error while loading shared libraries: libhts.so.3: cannot open shared object file: No such file or directory


In [ ]:
import sys
print(sys.getsizeof(sim_data[0]))

# Test the running time and size of core steps

In [ ]:
sim_cmd = ['/home/wbguo/iproject/BSReadSim/WGSIM/wgsim', 
           '-1', '100', '-2', '100','-e','0.005','-d','400','-s','25',
           '-r','0.1', '-N','1000',
           '-R','0.15','-X','0.15',
           '-S','2022',
           '-A','0.05','-h','0', '-m', '1', '/home/wbguo/iproject/BSReadSim/test/ref/BSB_test.fa']

In [ ]:
" ".join(sim_cmd)

In [ ]:
wgsim = subprocess.Popen(sim_cmd, stdout=subprocess.PIPE, universal_newlines=True)
sim_iter = iter(wgsim.stdout.readline, b'')

In [ ]:
def get_line(sim_iter):
    try:
        line = next(sim_iter).strip()
    except StopIteration:
        print("End of output\n")
        return None
    else:
        return line

In [ ]:
def process_variant_line(line: str) -> Dict:
    line_split = line.split('\t')

    try:
        chrom, pos, ref, alt, heter_flag = line_split
    except ValueError:
        return dict(chrom=line_split[0], pos = None)
    else:
        heter = heter_flag == '+'
        indel = int(ref == '-') - int(alt == '-') # 1 for ref=='-', -1 for alt=='-', o.w. 0
        offset= indel * max(len(ref), len(alt))
        if indel:
            iupac  = None
        else:
            iupac  = retrieve_iupac(alt)
            alt    = list(set(iupac) - set(ref))[0]
        return dict(chrom=chrom, pos=int(pos), ref=ref, alt=alt,
                    offset=offset, heter=heter, indel=indel, iupac=iupac)


def collect_variants(sim_iter):
    variant_dict = {}

    while True:
        line = get_line(sim_iter)
        if line == 'Contig Variant End':
            return variant_info['chrom'], variant_dict

        variant_info = process_variant_line(line)
        if variant_info['pos']:
            assert variant_info['pos'] not in variant_dict
            variant_dict[variant_info['pos']] = variant_info

In [ ]:
v = collect_variants(sim_iter)

In [ ]:
v

In [ ]:
from pympler import asizeof

In [ ]:
print(asizeof.asizeof(v))

In [ ]:
len(v[1])

In [ ]:
while True:
    x = get_line(sim_iter)
    if x[0] == "@":
        break

y = get_line(sim_iter)
z = get_line(sim_iter)
t = get_line(sim_iter)

In [ ]:
x

In [ ]:
x.split(' ')

In [ ]:
y

In [ ]:
z

In [ ]:
t

In [ ]:
np.frombuffer(t.encode('utf-8'), np.int8)

In [ ]:
#### 12 us & 640 byte for a read, used the fputc for output
def process_read_name(line_list: list):
    read_id, pair, flag_pos, flag_mut, flag_indel, qual, cgr = line_list[0].split(' ')
    cgr = np.frombuffer(cgr.encode(), dtype=np.int8)
    seq = np.frombuffer(line_list[1].encode(), dtype=np.int8)
    _, start, end, cover_pos, n_sub, n_indel, insrt_len, insrt_len2, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = np.frombuffer(line_list[3].encode(), np.int8)
    return dict(read_id=read_id, pair=int(pair), flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos), n_sub=int(n_sub), n_indel=int(n_indel), 
                insrt_len=int(insrt_len), insrt_len2=int(insrt_len2), qual = int(qual),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
%timeit process_read_name([x,y,z,t]) 

In [ ]:
import sys
obj = process_read_name([x,y,z,t])
sys.getsizeof(obj)

In [ ]:
obj

In [ ]:
chr(obj['qual'])

In [ ]:
''.join(['ACGT'[i] for i in obj['seq']])

In [ ]:
while True:
    x1 = get_line(sim_iter)
    if x1[0] == "@":
        break

y1 = get_line(sim_iter)
z1 = get_line(sim_iter)
t1 = get_line(sim_iter)

In [ ]:
obj1 = process_read_name([x1,y1,z1,t1])

In [ ]:
obj1

In [ ]:
sim_data = [obj, obj1]

In [ ]:
sys.getsizeof(sim_data)

In [ ]:
print(asizeof.asizeof(sim_data))

In [ ]:
%timeit obj1['start'] + np.arange(len(obj1['seq']))

In [ ]:
x = obj1['start'] + np.arange(len(obj1['seq']))

In [ ]:
%timeit x + obj1['ofs']

# Speed & memory test result

In [ ]:
#### 25 us & 640 byte for a read, used the %d for output
ascii_idx = np.array([i for i in range(48,58)] + [i for i in range(97, 103)])
ascii_val = np.array([i for i in range(0,16)])
ascii_arr = np.full(127, -1).astype(np.int8)
ascii_arr[ascii_idx] = ascii_val

def process_read_name2(line_list: list, read_len: int):
    read_id, pair, flag_pos, flag_mut, flag_indel, cgr = line_list[0].split(' ')
    cgr = np.bitwise_and(np.frombuffer(cgr.encode(), dtype=np.int8), 0x03)
    seq = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, start, end, cover_pos, n_sub, n_indel, insrt_len, insrt_len2, ofs= line_list[2].split(':')
    ofs = np.fromstring(ofs, dtype=np.int8, sep = ',')
    ctx = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id=read_id, pair=int(pair), flag_pos=int(flag_pos), flag_mut=int(flag_mut), flag_indel=int(flag_indel),
                start=int(start), end=int(end), cover_pos=int(cover_pos), n_sub=int(n_sub), n_indel=int(n_indel), 
                insrt_len=int(insrt_len), insrt_len2=int(insrt_len2),
                cgr=cgr, seq=seq, ofs=ofs, ctx=ctx)

In [ ]:
#### 38 us & 360 byte for a read, used the %d for output
def process_read_name3(line_list: list, read_len: int):
    arr = np.full([4, read_len], np.NaN)
    read_id, pair, num_var, num_indel, start, end, mut = line_list[0].split(' ')
    arr[0] = np.bitwise_and(np.frombuffer(mut.encode(), dtype=np.int8), 0x03)
    arr[1] = np.bitwise_and(np.frombuffer(line_list[1].encode(), dtype=np.int8), 0x03)
    _, n_sub, n_indel, insrt_len, ofs = line_list[2].split(':')
    arr[2] = np.fromstring(ofs, dtype=np.int8, sep = ',')
    arr[3] = ascii_arr[np.frombuffer(line_list[3].encode(), np.int8)]
    return dict(read_id = read_id, pair = int(pair),  start=int(start), end=int(end), 
                num_var = int(num_var), num_indel = int(num_indel), n_sub = int(n_sub), n_indel = int(n_indel),
                arr = arr)

In [ ]:
### if use %d
%timeit np.fromstring(y, dtype=np.int8)                               # 1.56 us, will give 48-51
%timeit np.frombuffer(y.encode(), dtype=np.int8)                      # 0.9  us, will give 48-51
%timeit np.array(list(y), dtype=np.int8)                              # 13.5 us, will give 0-3
%timeit np.frombuffer(y.encode(), dtype=np.int8) - 48                 # 4.25 us, will give 0-3
%timeit np.bitwise_and(np.frombuffer(y.encode(), dtype=np.int8), 0x3) # 4.38 us, will give 0-3
%timeit ascii_arr[np.frombuffer(y.encode(), dtype=np.int8)]           # 4.8  us, will give 0-3